# Cross-Currency Basis Swap Pricing with fi-claude

A **cross-currency basis swap** exchanges floating-rate payments in two different currencies.
In this notebook we price a **USD/EUR** swap where:

- The **near leg** pays USD SOFR (flat)
- The **far leg** pays EUR EURIBOR 3M minus 15 basis points
- Both legs exchange notionals at inception and maturity

The -15 bps spread on the EUR leg is the *basis* — it reflects the relative
cost of borrowing EUR versus USD in the cross-currency market.

**Key design point:** every calculation in fi-claude is a **pure function** over
**immutable data**. There is no hidden state, no mutation, no I/O inside the
pricing layer. The same inputs always produce the same outputs.

## 1. Build Market Data

We construct realistic discount curves for USD and EUR, plus an FX spot rate
and forward curve. All market data is captured in a single immutable
`MarketData` snapshot.

In [ ]:
from datetime import date
from decimal import Decimal

from fi_claude.data.common import Currency, DayCountConvention
from fi_claude.data.curves import CurveNode, DiscountCurve, FxForwardCurve
from fi_claude.data.market import MarketData

# --- USD discount curve (~4.5% rates) ---
usd_nodes = (
    CurveNode(date=date(2025, 9, 15),  value=0.9870),   # 3M
    CurveNode(date=date(2025, 12, 15), value=0.9740),   # 6M
    CurveNode(date=date(2026, 6, 15),  value=0.9560),   # 1Y
    CurveNode(date=date(2027, 6, 15),  value=0.9140),   # 2Y
    CurveNode(date=date(2028, 6, 15),  value=0.8720),   # 3Y
    CurveNode(date=date(2030, 6, 15),  value=0.7920),   # 5Y
    CurveNode(date=date(2032, 6, 15),  value=0.7160),   # 7Y
    CurveNode(date=date(2035, 6, 15),  value=0.6100),   # 10Y
    CurveNode(date=date(2045, 6, 15),  value=0.3800),   # 20Y
    CurveNode(date=date(2055, 6, 15),  value=0.2500),   # 30Y
)
usd_curve = DiscountCurve(
    reference_date=date(2025, 6, 15),
    currency=Currency.USD,
    nodes=usd_nodes,
)

# --- EUR discount curve (~3.5% rates) ---
eur_nodes = (
    CurveNode(date=date(2025, 9, 15),  value=0.9920),   # 3M
    CurveNode(date=date(2025, 12, 15), value=0.9830),   # 6M
    CurveNode(date=date(2026, 6, 15),  value=0.9660),   # 1Y
    CurveNode(date=date(2027, 6, 15),  value=0.9330),   # 2Y
    CurveNode(date=date(2028, 6, 15),  value=0.9010),   # 3Y
    CurveNode(date=date(2030, 6, 15),  value=0.8390),   # 5Y
    CurveNode(date=date(2032, 6, 15),  value=0.7800),   # 7Y
    CurveNode(date=date(2035, 6, 15),  value=0.6900),   # 10Y
    CurveNode(date=date(2045, 6, 15),  value=0.4700),   # 20Y
    CurveNode(date=date(2055, 6, 15),  value=0.3200),   # 30Y
)
eur_curve = DiscountCurve(
    reference_date=date(2025, 6, 15),
    currency=Currency.EUR,
    nodes=eur_nodes,
)

# --- FX: EUR/USD spot and forward curve ---
fx_spot = 1.0870  # 1 EUR = 1.0870 USD

fx_forward_curve = FxForwardCurve(
    reference_date=date(2025, 6, 15),
    base_currency=Currency.EUR,
    quote_currency=Currency.USD,
    spot_rate=fx_spot,
    nodes=(
        CurveNode(date=date(2025, 9, 15),  value=-0.0025),   # 3M fwd pts
        CurveNode(date=date(2025, 12, 15), value=-0.0048),   # 6M
        CurveNode(date=date(2026, 6, 15),  value=-0.0095),   # 1Y
        CurveNode(date=date(2027, 6, 15),  value=-0.0185),   # 2Y
    ),
)

# --- Assemble the immutable MarketData snapshot ---
market = MarketData(
    valuation_date=date(2025, 6, 15),
    discount_curves={
        "USD": usd_curve,
        "EUR": eur_curve,
    },
    fx_spot_rates={
        "EUR/USD": fx_spot,
    },
    fx_curves={
        "EUR/USD": fx_forward_curve,
    },
)

# --- Summary ---
print("Market Data Summary")
print("=" * 50)
print(f"Valuation date: {market.valuation_date}")
print(f"EUR/USD spot:   {fx_spot}")
print()
print(f"USD discount curve ({len(usd_nodes)} nodes):")
for n in usd_nodes:
    tenor_yrs = (n.date - date(2025, 6, 15)).days / 365.25
    print(f"  {n.date}  df={n.value:.4f}  (~{tenor_yrs:.1f}Y)")
print()
print(f"EUR discount curve ({len(eur_nodes)} nodes):")
for n in eur_nodes:
    tenor_yrs = (n.date - date(2025, 6, 15)).days / 365.25
    print(f"  {n.date}  df={n.value:.4f}  (~{tenor_yrs:.1f}Y)")

## 2. Define the Swap

We create a 2-year USD/EUR cross-currency basis swap with:
- **Near leg (USD):** 10M notional, SOFR flat, quarterly ACT/360
- **Far leg (EUR):** 9.2M notional, EURIBOR 3M - 15 bps, quarterly ACT/360
- Initial and final notional exchanges (the defining feature of an xccy swap)
- Mark-to-market resets on the near leg
- FX rate at inception: 1.087 (consistent with 10M USD vs 9.2M EUR)

In [ ]:
from fi_claude.data.instruments import XccyBasisSwap, XccyLeg
from fi_claude.data.common import CompoundingMethod

swap = XccyBasisSwap(
    near_leg=XccyLeg(
        currency=Currency.USD,
        notional=Decimal("10000000"),       # 10M USD
        floating_index="SOFR",
        spread_bps=0.0,                     # flat SOFR
        day_count=DayCountConvention.ACT_360,
        compounding=CompoundingMethod.COMPOUNDED,
        payment_frequency_months=3,
    ),
    far_leg=XccyLeg(
        currency=Currency.EUR,
        notional=Decimal("9200000"),        # 9.2M EUR
        floating_index="EURIBOR_3M",
        spread_bps=-15.0,                   # EURIBOR - 15 bps
        day_count=DayCountConvention.ACT_360,
        compounding=CompoundingMethod.COMPOUNDED,
        payment_frequency_months=3,
    ),
    start_date=date(2025, 6, 15),
    end_date=date(2027, 6, 15),
    initial_exchange=True,
    final_exchange=True,
    mark_to_market_reset=True,
    fx_rate_at_inception=1.087,
)

# --- Print swap terms ---
print("Swap Terms")
print("=" * 50)
print(f"Type:           Cross-Currency Basis Swap")
print(f"Start date:     {swap.start_date}")
print(f"End date:       {swap.end_date}")
print(f"Tenor:          2 years")
print(f"FX at inception:{swap.fx_rate_at_inception}")
print(f"Init exchange:  {swap.initial_exchange}")
print(f"Final exchange: {swap.final_exchange}")
print(f"MtM reset:      {swap.mark_to_market_reset}")
print()
print("Near Leg (USD):")
print(f"  Notional:  {swap.near_leg.notional:>15,}")
print(f"  Index:     {swap.near_leg.floating_index}")
print(f"  Spread:    {swap.near_leg.spread_bps:+.0f} bps")
print(f"  Frequency: Q ({swap.near_leg.payment_frequency_months}M)")
print(f"  Day count: {swap.near_leg.day_count.value}")
print()
print("Far Leg (EUR):")
print(f"  Notional:  {swap.far_leg.notional:>15,}")
print(f"  Index:     {swap.far_leg.floating_index}")
print(f"  Spread:    {swap.far_leg.spread_bps:+.0f} bps")
print(f"  Frequency: Q ({swap.far_leg.payment_frequency_months}M)")
print(f"  Day count: {swap.far_leg.day_count.value}")

## 3. Price the Swap

The pricer is a pure function: `(XccyBasisSwap, MarketData) -> PricingResult`.
It computes the PV of each leg, the notional exchange PV, and returns a
complete cashflow schedule.

In [ ]:
from fi_claude.pricers.xccy_basis_swap import price_xccy_basis_swap

result = price_xccy_basis_swap(swap, market)

print("Pricing Result")
print("=" * 50)
print(f"Instrument:     {result.instrument_type}")
print(f"Valuation date: {result.valuation_date}")
print(f"Currency:       {result.currency.value}")
print(f"Present Value:  {result.present_value:>15,.2f}")
print()
print("Decomposition:")
print(f"  Near leg PV (USD):        {result.details['near_leg_pv']:>15,.2f}")
print(f"  Far leg PV (domestic):    {result.details['far_leg_pv_domestic']:>15,.2f}")
print(f"  Far leg PV (EUR):         {result.details['far_leg_pv_foreign']:>15,.2f}")
print(f"  Notional exchange PV:     {result.details['exchange_pv']:>15,.2f}")
print(f"  FX spot (EUR/USD):        {result.details['fx_spot']:>15.4f}")
print()
print(f"Total PV = near_pv - far_pv_domestic + exchange_pv")
print(f"         = {result.details['near_leg_pv']:,.2f} - {result.details['far_leg_pv_domestic']:,.2f} + {result.details['exchange_pv']:,.2f}")
print(f"         = {result.present_value:,.2f}")

## 4. Cashflow Schedule

The pricer returns every projected cashflow with date, amount, and currency.
USD and EUR cashflows interleave since both legs pay quarterly on the same dates.

In [ ]:
# Print the cashflow schedule as a formatted table
cashflows = result.cashflows

print("Cashflow Schedule")
print("=" * 55)
print(f"{'Date':<14} {'Currency':<10} {'Amount':>18}")
print("-" * 55)

usd_total = 0.0
eur_total = 0.0
for cf in sorted(cashflows, key=lambda c: (c.payment_date, c.currency.value)):
    amt = float(cf.amount)
    print(f"{cf.payment_date}   {cf.currency.value:<10} {amt:>18,.2f}")
    if cf.currency == Currency.USD:
        usd_total += amt
    else:
        eur_total += amt

print("-" * 55)
print(f"{'Total USD':<24} {usd_total:>18,.2f}")
print(f"{'Total EUR':<24} {eur_total:>18,.2f}")
print(f"\n{len(cashflows)} cashflows total")

## 5. Scenario Analysis -- Rate Shocks

We test three rate scenarios using fi-claude's composable shock framework:

1. **Fed +50 bps** -- parallel shift of the USD curve
2. **ECB -25 bps** -- parallel shift of the EUR curve
3. **Combined** -- both shocks applied together

Each shock produces a new immutable `MarketData`; repricing is a pure function call.

In [ ]:
from fi_claude.risk.shocks import scenario, parallel, fx_shock, apply_shocks

base_pv = float(result.present_value)

# Define three rate scenarios
scenarios = {
    "Fed +50 bps": scenario("Fed +50 bps", rates=parallel("USD", 50)),
    "ECB -25 bps": scenario("ECB -25 bps", rates=parallel("EUR", -25)),
    "Combined":    scenario("Combined",
                       rates=(parallel("USD", 50), parallel("EUR", -25))),
}

print("Rate Shock Scenarios")
print("=" * 60)
print(f"{'Scenario':<20} {'Base PV':>12} {'Shocked PV':>12} {'PV Change':>12}")
print("-" * 60)

for name, shock_set in scenarios.items():
    shocked_market = apply_shocks(market, shock_set)
    shocked_result = price_xccy_basis_swap(swap, shocked_market)
    shocked_pv = float(shocked_result.present_value)
    change = shocked_pv - base_pv
    print(f"{name:<20} {base_pv:>12,.2f} {shocked_pv:>12,.2f} {change:>+12,.2f}")

print("-" * 60)
print("\nNote: USD rate hikes increase the value of receiving USD SOFR.")
print("EUR rate cuts reduce the value of paying EUR EURIBOR.")

## 6. Scenario Analysis -- FX Shocks

Cross-currency swaps have significant FX exposure due to notional exchanges.
Here we shock EUR/USD by -5% and +5% to see the PV impact.

In [ ]:
fx_scenarios = {
    "EUR/USD -5%": scenario("EUR/USD -5%", fx=fx_shock("EUR/USD", pct=-5.0)),
    "EUR/USD +5%": scenario("EUR/USD +5%", fx=fx_shock("EUR/USD", pct=+5.0)),
}

print("FX Shock Scenarios")
print("=" * 65)
print(f"{'Scenario':<16} {'FX Spot':>10} {'Base PV':>12} {'Shocked PV':>12} {'Change':>12}")
print("-" * 65)

for name, shock_set in fx_scenarios.items():
    shocked_market = apply_shocks(market, shock_set)
    new_fx = shocked_market.fx_spot_rates["EUR/USD"]
    shocked_result = price_xccy_basis_swap(swap, shocked_market)
    shocked_pv = float(shocked_result.present_value)
    change = shocked_pv - base_pv
    print(f"{name:<16} {new_fx:>10.4f} {base_pv:>12,.2f} {shocked_pv:>12,.2f} {change:>+12,.2f}")

print("-" * 65)
print(f"\nBase EUR/USD spot: {fx_spot:.4f}")
print("The FX sensitivity is large because the notional exchanges")
print("convert EUR cashflows at the shocked spot rate.")

## 7. Seasoning Analysis

Seasoning decomposes the P&L from the passage of time into:
- **Carry**: income from cashflows received in the period
- **Rolldown**: PV change from aging along the curve (shorter tenor = higher DF)
- **Total theta**: carry + rolldown

We also track DV01 evolution -- how rate sensitivity drifts over time.

In [ ]:
from functools import partial
from fi_claude.risk.seasoning import season_portfolio

# Bind the swap into the pricer so it matches the (MarketData) -> PricingResult signature
pricer = lambda m: price_xccy_basis_swap(swap, m)

report = season_portfolio(
    pricer=pricer,
    market=market,
    horizons=(1, 7, 30),
    rate_curve_keys=("USD", "EUR"),
)

print("Seasoning Report")
print("=" * 70)
print(f"Instrument: {report.instrument_type}")
print(f"Currency:   {report.currency}")
print()

# P&L decomposition table
print(f"{'Horizon':<10} {'Theta':>12} {'Carry':>12} {'Rolldown':>12} {'Ann. Theta':>12}")
print("-" * 60)
for h in report.horizons:
    print(f"{h.horizon_label:<10} {h.total_theta:>12,.2f} {h.carry:>12,.2f} "
          f"{h.rolldown:>12,.2f} {h.theta_annual:>12,.2f}")

# DV01 evolution
print()
print("DV01 Evolution (per 1bp parallel shift)")
print("-" * 60)
print(f"{'Horizon':<10} {'USD DV01':>12} {'EUR DV01':>12} {'USD chg':>12} {'EUR chg':>12}")
print("-" * 60)
for h in report.horizons:
    usd_base = h.base_risk.dv01.get("USD", 0.0)
    eur_base = h.base_risk.dv01.get("EUR", 0.0)
    usd_chg = h.dv01_change.get("USD", 0.0)
    eur_chg = h.dv01_change.get("EUR", 0.0)
    print(f"{h.horizon_label:<10} {usd_base:>12,.4f} {eur_base:>12,.4f} "
          f"{usd_chg:>+12,.4f} {eur_chg:>+12,.4f}")

## 8. Sensitivity Heatmap: USD Rates x FX

A 2D grid showing how the swap PV changes across simultaneous USD rate
shocks (-50 to +50 bps) and EUR/USD FX shocks (-5% to +5%).

This reveals the cross-gamma between rates and FX -- a key risk for
xccy basis swap books.

In [ ]:
rate_bumps = [-50, -25, 0, 25, 50]       # bps
fx_bumps   = [-5.0, -2.5, 0.0, 2.5, 5.0]  # percent

# Compute the grid
grid = {}
for rb in rate_bumps:
    for fb in fx_bumps:
        shocks = []
        rates_arg = parallel("USD", rb) if rb != 0 else None
        fx_arg = fx_shock("EUR/USD", pct=fb) if fb != 0.0 else None

        s = scenario(
            f"USD {rb:+d}bp / FX {fb:+.1f}%",
            rates=rates_arg if rates_arg else (),
            fx=fx_arg if fx_arg else (),
        )
        shocked_mkt = apply_shocks(market, s)
        r = price_xccy_basis_swap(swap, shocked_mkt)
        grid[(rb, fb)] = float(r.present_value) - base_pv

# Print the heatmap
print("PV Change Heatmap (USD rate bps x EUR/USD FX %)")
print("=" * 72)

# Header row
header = f"{'USD bps \\\\ FX%':<14}"
for fb in fx_bumps:
    header += f"{fb:>+10.1f}%"
print(header)
print("-" * 72)

for rb in rate_bumps:
    row = f"{rb:>+6d} bps      "
    for fb in fx_bumps:
        val = grid[(rb, fb)]
        row += f"{val:>+10,.0f}"
    print(row)

print("-" * 72)
print("\nValues are PV changes from the base case in USD.")
print("Diagonal moves (rate up + FX down) compound the risk.")

## 9. Conclusion

**Key observations from this demo:**

1. **Notional exchanges dominate**: The PV is mostly driven by the initial and final
   notional exchanges, not the floating coupon differential. This is characteristic
   of xccy basis swaps.

2. **FX sensitivity is large**: A 5% EUR/USD move causes a much bigger PV change than
   a 50 bps rate shock. The notional exchange at maturity creates significant FX delta.

3. **Rate and FX risks interact**: The heatmap shows non-trivial cross-gamma --
   simultaneous adverse moves in rates and FX compound the loss.

4. **Theta is asymmetric**: As the swap ages, the remaining notional exchange
   discount factor increases (approaches 1.0), which dominates the theta profile.

5. **Reproducibility**: Every result in this notebook is deterministic. The same
   `MarketData` and `XccyBasisSwap` will always produce the same `PricingResult`,
   because every pricer is a pure function over immutable data.

For the mathematical details behind the pricing, see `docs/math.md`.